In [1]:
#Libraries Required
from pathlib import Path
import re
import json

In [2]:
#Reading the Raw txt file
RAW_TXT = Path("/Users/varunchandrashekar/Tenantmate/code/data/processed/act_raw.txt")

def load_raw_text():
    return RAW_TXT.read_text(encoding="utf-8")

# inspect: how does it actually look?
text = load_raw_text()
print(text[:2000])      # first chunk — see the front matter
print("----")
print(len(text), "characters")

Residential Tenancies Act 2010 No 42
[2010-42]
NNeeww SSoouutthh WWaalleess
Status Information
Currency of version
Current version for 15 August 2025 to date (accessed 23 May 2026 at 17:57)
Legislation on this site is usually updated within 3 working days after a change to the legislation.
Provisions in force
The provisions displayed in this version of the legislation have all commenced.
Notes—
• Does not include amendments by
Victims Rights and Victims of Crime Commissioner Act 2025No 64(not commenced)
Residential Tenancies Amendment (Domestic Violence Reform) Act 2025No 65(not commenced)
• See also
Residential Tenancies Amendment (Animals in Residential Premises) Bill 2024[Non-government Bill—
the Hon Emma Hurst, MLC]
Residential Tenancies Amendment (Protection of Personal Information) Bill 2025
Fair Trading and Building Legislation Amendment Bill 2026
Statute Law (Miscellaneous Provisions) Bill 2026
Responsible Minister
• Minister for Better Regulation and Fair Trading
• Minister fo

In [3]:
#Defining Boiler Plate and removal
PAGE_RE = re.compile(r"^Page \d+ of \d+$")

def _is_boilerplate(line):
    if PAGE_RE.match(line):
        return True
    if line.startswith("Current version for"):
        return True
    if "[NSW]" in line and "Residential Tenancies Act 2010" in line:
        return True
    return False

def strip_boilerplate(text):
    text = text.replace("\f", "\n")  #split glued page-break
    lines = text.split("\n")
    return "\n".join(ln for ln in lines if not _is_boilerplate(ln.strip()))

\f (form-feed) is an invisible character that PDF extractors insert to mark "this is where one page ends and the next begins." Your eyes never see it, but it's sitting there in the text between page 1's content and page 2's content.
When pdfplumber extracted each page separately, you joined them with \f as the separator — so the page boundaries are preserved in the final text. That's why we have to handle it now.

In [4]:
#We are making the form feed breaker to line breakers for easier breaking down
prepped = text.replace("\f", "\n")
print(type(prepped))
print(repr(prepped[:1500]))
print("Total lines:", len(prepped.split("\n")))

<class 'str'>
"Residential Tenancies Act 2010 No 42\n[2010-42]\nNNeeww SSoouutthh WWaalleess\nStatus Information\nCurrency of version\nCurrent version for 15 August 2025 to date (accessed 23 May 2026 at 17:57)\nLegislation on this site is usually updated within 3 working days after a change to the legislation.\nProvisions in force\nThe provisions displayed in this version of the legislation have all commenced.\nNotes—\n• Does not include amendments by\nVictims Rights and Victims of Crime Commissioner Act 2025No 64(not commenced)\nResidential Tenancies Amendment (Domestic Violence Reform) Act 2025No 65(not commenced)\n• See also\nResidential Tenancies Amendment (Animals in Residential Premises) Bill 2024[Non-government Bill—\nthe Hon Emma Hurst, MLC]\nResidential Tenancies Amendment (Protection of Personal Information) Bill 2025\nFair Trading and Building Legislation Amendment Bill 2026\nStatute Law (Miscellaneous Provisions) Bill 2026\nResponsible Minister\n• Minister for Better Regula

Preview lines flagged as boilerplate
Dry-run of the boilerplate filter: lists which lines would be dropped, without modifying the text. Lets us verify the filter targets repeating page furniture (headers, footers, page numbers) and not real content.

In [5]:
dropped = [ln for ln in prepped.split("\n") if _is_boilerplate(ln.strip())]
print(f"Flagged: {len(dropped)}")
for ln in dropped[:20]:
    print(repr(ln))

Flagged: 337
'Current version for 15 August 2025 to date (accessed 23 May 2026 at 17:57)'
'Residential Tenancies Act 2010 No 42 [NSW]'
'Current version for 15 August 2025 to date (accessed 23 May 2026 at 17:57) Page 2 of 169'
'Residential Tenancies Act 2010 No 42 [NSW]'
'Current version for 15 August 2025 to date (accessed 23 May 2026 at 17:57) Page 3 of 169'
'Residential Tenancies Act 2010 No 42 [NSW]'
'Current version for 15 August 2025 to date (accessed 23 May 2026 at 17:57) Page 4 of 169'
'Residential Tenancies Act 2010 No 42 [NSW]'
'Current version for 15 August 2025 to date (accessed 23 May 2026 at 17:57) Page 5 of 169'
'Residential Tenancies Act 2010 No 42 [NSW]'
'Current version for 15 August 2025 to date (accessed 23 May 2026 at 17:57) Page 6 of 169'
'Residential Tenancies Act 2010 No 42 [NSW]'
'Current version for 15 August 2025 to date (accessed 23 May 2026 at 17:57) Page 7 of 169'
'Residential Tenancies Act 2010 No 42 [NSW]'
'Current version for 15 August 2025 to date (acce

Runs strip_boilerplate on the full text and confirms the header line (Residential Tenancies Act 2010 No 42 [NSW]) no longer appears — a quick sanity check that the filter worked end to end.

In [6]:
cleaned = strip_boilerplate(text)
nsw_lines = [ln for ln in cleaned.split("\n") if "[NSW]" in ln]
print(f"[NSW] lines remaining: {len(nsw_lines)}")
for ln in nsw_lines[:5]:
    print(repr(ln))

[NSW] lines remaining: 0


In [7]:
# Goal: is to slice off pages 1-13 (status info + table of contents) so your text starts at the actual Act body
def drop_front_mattter(text):
    marker = "An Act with respect to"
    idx = text.rfind(marker)  #rfind = last occurance, skips the TOC(Table of Contents) entry
    if idx == -1:
        raise ValueError("Start marker not found")
    return text[idx:]

body = drop_front_mattter(cleaned)

print(repr(body[:500]))
print("---")
print(f"Body length: {len(body):,} chars (was {len(cleaned):,})")

'An Act with respect to the rights and obligations of landlords and tenants, rents, rental bonds\nand other matters relating to residential tenancy agreements; and for other purposes.\nPart 1 Preliminary\nDivision 1 General\n1 Name of Act\nThis Act is the Residential Tenancies Act 2010.\n2 Commencement\nThis Act commences on a day or days to be appointed by proclamation.\n3 Definitions\n(1) In this Act—\nacceptable behaviour agreement—see section 138.\napprehended violence order has the same meaning as it h'
---
Body length: 308,177 chars (was 356,836)


In [12]:
SECTION_RE  = re.compile(r"^(\d+[A-Z]?)\s+([A-Z].*)$")
PART_RE     = re.compile(r"^Part\s+\d+\s+(.+)$")
DIVISION_RE = re.compile(r"^Division\s+\d+\s+(.+)$")
SCHEDULE_RE = re.compile(r"^Schedule\s+\d+\s*(.*)$")

def split_sections(body_text):
    sections = []
    current = None
    current_part = None
    current_division = None
    current_schedule = None

    for line in body_text.split("\n"):
        stripped = line.strip()

        # Schedule resets Part and Division (Schedules have their own hierarchy)
        if SCHEDULE_RE.match(stripped):
            current_schedule = stripped
            current_part = None
            current_division = None
            continue

        if PART_RE.match(stripped):
            current_part = stripped
            continue
        if DIVISION_RE.match(stripped):
            current_division = stripped
            continue

        m = SECTION_RE.match(stripped)
        if m:
            sec_num_str = m.group(1)
            digits = ''.join(c for c in sec_num_str if c.isdigit())

            # NEW: reject false positives — real sections are at most 3 digits
            # (largest real section is ~230 + optional letter suffix).
            # 4+ digits = a year masquerading as a section heading (e.g. "1977 Act").
            if len(digits) > 3:
                if current is not None:
                    current["text"] += line + "\n"
                continue

            if current is not None:
                sections.append(current)
            current = {
                "section_number": sec_num_str,
                "section_title": m.group(2).strip(),
                "part": current_part,
                "division": current_division,
                "schedule": current_schedule,
                "text": "",
            }
        else:
            if current is not None:
                current["text"] += line + "\n"

    if current is not None:
        sections.append(current)

    return sections

sections = split_sections(body)
print(f"Total sections: {len(sections)}")

in_schedule = sum(1 for s in sections if s["schedule"])
print(f"In main Act: {len(sections) - in_schedule}")
print(f"In Schedules: {in_schedule}")

Total sections: 328
In main Act: 276
In Schedules: 52


In [13]:
# Did letter-suffix sections (8A, 55A, 222A) get caught?
suffixed = [s for s in sections if any(c.isalpha() for c in s['section_number'])]
print(f"Letter-suffix sections: {len(suffixed)}")
for s in suffixed[:5]:
    print(f"  {s['section_number']}  {s['section_title']}")

# Spot-check a section's body
sec3 = next(s for s in sections if s['section_number'] == '3')
print(f"\nSection 3 ({sec3['section_title']}):")
print(f"  Part: {sec3['part']}")
print(f"  Division: {sec3['division']}")
print(f"  Body length: {len(sec3['text'])} chars")
print(f"  First 200 chars: {sec3['text'][:200]}")

Letter-suffix sections: 63
  8A  Application of Act to premises
  22A  Prohibition on certain matters relating to advertising or soliciting amounts of rent
  31A  Landlord’s information statement
  54A  Limit on liability of tenant for actions of other tenants occurring during domestic
  55A  Publishing photographs of residential premises with tenant’s consent

Section 3 (Definitions):
  Part: Part 1 Preliminary
  Division: Division 1 General
  Body length: 4952 chars
  First 200 chars: (1) In this Act—
acceptable behaviour agreement—see section 138.
apprehended violence order has the same meaning as it has in the Crimes
(Domestic and Personal Violence) Act 2007 and includes a provis


In [14]:
OUT_JSONL = Path("/Users/varunchandrashekar/Tenantmate/code/data/processed/nsw_sections.jsonl")

META = {
    "jurisdiction": "NSW",
    "act": "Residential Tenancies Act 2010 No 42",
    "source_url": "https://legislation.nsw.gov.au/view/whole/pdf/inforce/current/act-2010-042",
    "version": "current as at 15 August 2025",
    "downloaded_date": "2026-05-23",
}

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for s in sections:
        if s["schedule"]:
            sched_tag = s["schedule"].split()[0] + s["schedule"].split()[1]   # "Schedule2"
            chunk_id = f"NSW-RTA2010-{sched_tag}-s{s['section_number']}"
        else:
            chunk_id = f"NSW-RTA2010-s{s['section_number']}"

        record = {
            "chunk_id": chunk_id,
            "text": s["text"].strip(),
            "section_number": s["section_number"],
            "section_title": s["section_title"],
            "part": s["part"],
            "division": s["division"],
            "schedule": s["schedule"],
            **META,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# Verify uniqueness
with open(OUT_JSONL) as f:
    chunks = [json.loads(line) for line in f]

ids = [c["chunk_id"] for c in chunks]
print(f"Unique IDs: {len(set(ids))} / {len(ids)}")

Unique IDs: 328 / 328


In [15]:
#Verfiying it
with open(OUT_JSONL) as f:
    lines = f.readlines()

print(f"Total records: {len(lines)}")
print("\nFirst record:")
print(json.dumps(json.loads(lines[0]), indent=2, ensure_ascii=False))

Total records: 328

First record:
{
  "chunk_id": "NSW-RTA2010-s1",
  "text": "This Act is the Residential Tenancies Act 2010.",
  "section_number": "1",
  "section_title": "Name of Act",
  "part": "Part 1 Preliminary",
  "division": "Division 1 General",
  "schedule": null,
  "jurisdiction": "NSW",
  "act": "Residential Tenancies Act 2010 No 42",
  "source_url": "https://legislation.nsw.gov.au/view/whole/pdf/inforce/current/act-2010-042",
  "version": "current as at 15 August 2025",
  "downloaded_date": "2026-05-23"
}
